## Map-Reduce 요약

##### 각 문서를 Map 단계에서 개별적으로 요약, Reduce 단계에서 요약본들을 최종 요약 본으로 합침

- 대규모 문서를 언어 모델의 토큰 제한에 구애받지 않고 효율적으로 요약
- Map: 문서를 작은청크로 나누는 단계
- Reduce: 각 청크의 요약을 결합

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI



llm = ChatOpenAI(
    temperature=0,
    model_name = "gpt-4o-mini"
)

docs = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf").load()
docs = docs[3:8] #문서 일부만 요약

#map 단계 - 각각의 문서를 개별적으로 요약
map_prompt = ChatPromptTemplate.from_template("""
다음 문서 내용을 요약하세요.

문서:
{context}
""")

map_chain = map_prompt|llm|StrOutputParser()

#document 객체에서 실제 텍스트만 추출
doc_contents = [
    doc.page_content
    for doc in docs
]

#5개 문서를 각각 요약
docs_summaries = map_chain.batch(
    [{"context":content} for content in doc_contents]
)

#reduce 단계 - 각각의 요약을 하나로 통합
reduce_prompt = ChatPromptTemplate.from_template("""
다음은 문서를 요약한 결과입니다.

{context}

위 요약들을 종합하여 하나의 최종 요약을 작성하세요.
한국어로 답변하세요.
""")

reduce_chain = reduce_prompt | llm | StrOutputParser()

#여러 요약을 하나의 문자열로 합치기
combined_summary = "\n\n".join(docs_summaries)

#reduce 실행
#최종 요약
answer = reduce_chain.invoke(
    {"context": combined_summary}
)


print(answer)


2023년 10월 30일, 미국 바이든 대통령은 '안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령'에 서명하여 AI의 안전, 보안, 개인정보 보호, 형평성, 소비자 보호, 노동자 지원, 혁신 촉진, 국제 협력을 위한 여러 조치를 명시했습니다. 이 행정명령은 기업이 AI 시스템의 안전 테스트 결과를 정부와 공유하고, 차별적 사용을 방지하기 위한 법률을 마련하는 등의 내용을 포함하고 있습니다.

같은 날, G7은 '히로시마 AI 프로세스'를 통해 AI 시스템 개발을 위한 국제 행동강령에 합의하였으며, AI 위험 식별과 완화를 위한 자발적 채택을 권장하고, AI 수명주기 전반에 걸친 위험 평가와 투명성 보장을 요구했습니다. 이어서 11월 1일부터 2일에는 영국 블레츨리 파크에서 AI 안전성 정상회의가 열려 28개국이 '블레츨리 선언'을 발표하고, AI 안전성을 높이기 위한 협력을 다짐했습니다.

한편, 미국 캘리포니아 북부지방법원은 미드저니, 스태빌리티AI, 디비언트아트에 대한 저작권 침해 소송을 기각하였고, 미국 연방거래위원회(FTC)는 생성 AI의 소비자 보호와 경쟁 측면에서 우려를 표명하며 법적 조치를 취하겠다고 밝혔습니다. 이러한 일련의 사건들은 AI의 안전하고 신뢰할 수 있는 개발과 활용을 위한 국제적 노력과 법적 쟁점들을 반영하고 있습니다.
